# Pick Me Up, Infinite Gacha Scraper

This notebook contains scrapers for fetching the chapters of the *Pick Me Up: Infinite Gacha* light novel.

- **Part 1: Empire Novel Scraper** (Recommended): Fetches chapters 1 to 327. These are the actual chapters with full text.
- **Part 2: NovelBin Scraper**: Fetches chapters 1 to 400. (Note: NovelBin has Cloudflare protection; if running locally, you might need to copy your browser Cookie header into the requests setup if it blocks you with a 403).
- **Part 3: Legacy Novellunar Scraper**: Original scraper setup.

## Dependencies
Install the required libraries if you haven't already.

In [ ]:
# !pip install requests beautifulsoup4

## Part 1: Empire Novel Scraper (Chapters 1 - 327)

This scraper fetches chapters from `https://www.empirenovel.com/novel/pick-me-up/` and saves them in the `public/chapters` folder so the PWA reader can load them.

In [ ]:
import os
import time
import random
import requests
from bs4 import BeautifulSoup
import re

# Base URL for the novel chapters on Empire Novel
BASE_URL = "https://www.empirenovel.com/novel/pick-me-up/{}"

# Headers to mimic a browser and prevent bot blocking
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.empirenovel.com/novel/pick-me-up"
}

def clean_text(text):
    """
    Cleans up redundant whitespaces and consecutive empty lines.
    """
    lines = [line.strip() for line in text.split('\n')]
    cleaned_lines = []
    prev_was_empty = False
    for line in lines:
        if not line:
            if not prev_was_empty:
                cleaned_lines.append("")
                prev_was_empty = True
        else:
            cleaned_lines.append(line)
            prev_was_empty = False
    return "\n".join(cleaned_lines)

def fetch_chapter_empire(chapter_num, session):
    """
    Fetches a single chapter from Empire Novel and returns its title and parsed content.
    """
    url = BASE_URL.format(chapter_num)
    try:
        response = session.get(url, headers=HEADERS, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Locate the main novel content div
            read_novel_div = soup.find('div', id='read-novel')
            if not read_novel_div:
                print(f"Warning: No div with id='read-novel' found for chapter {chapter_num}")
                return None, None
            
            # Extract chapter title
            title_tag = read_novel_div.find('h3')
            title = title_tag.get_text(strip=True) if title_tag else f"Chapter {chapter_num}"
            
            # Extract paragraphs
            paragraphs = read_novel_div.find_all('p')
            if not paragraphs:
                print(f"Warning: No paragraphs found for chapter {chapter_num}")
                return None, None
            
            # Join paragraphs with double newlines
            raw_text = "\n\n".join([p.get_text().strip() for p in paragraphs])
            cleaned_content = clean_text(raw_text)
            
            return title, cleaned_content
        else:
            print(f"Failed to load chapter {chapter_num} - Status Code: {response.status_code}")
            return None, None
    except Exception as e:
        print(f"Error fetching chapter {chapter_num}: {e}")
        return None, None

# Configure and run the scraper
start_chapter = 1
end_chapter = 327  # Up to chapter 327
output_dir = os.path.join("public", "chapters")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

session = requests.Session()
print(f"Downloading chapters {start_chapter} to {end_chapter} from Empire Novel...")

for ch in range(start_chapter, end_chapter + 1):
    title, content = fetch_chapter_empire(ch, session)
    
    if title and content:
        # Create a filename-safe title
        safe_title = "".join([c for c in title if c.isalpha() or c.isdigit() or c==' ' or c=='-']).rstrip()
        safe_title = re.sub(r'\s+', ' ', safe_title)
        filename = f"{ch:03d}_{safe_title.replace(' ', '_')}.md"
        filepath = os.path.join(output_dir, filename)
        
        # Format as Markdown
        markdown_content = f"# {title}\n\n{content}\n"
        
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(markdown_content)
        print(f"Saved Chapter {ch}: {filename}")
    else:
        print(f"Skipped Chapter {ch} due to an error.")
        
    # Respectful delay between 1.0 and 2.5 seconds
    delay = random.uniform(1.0, 2.5)
    time.sleep(delay)

print("Finished downloading all chapters from Empire Novel!")

## Part 2: NovelBin Scraper (Chapters 1 - 400)

Fetches chapters from `https://novelbin.com/b/pick-me-up-infinite-gacha/chapter-{}`.

**Note**: If you get a `403 Forbidden` error (due to Cloudflare), open the site in your browser, open DevTools (F12), go to the Network tab, refresh the page, find a request to `novelbin.com`, copy the `Cookie` header value from your request headers, and add it to `HEADERS` below as `"Cookie": "<paste-cookies-here>"`.

In [ ]:
import os
import time
import random
import requests
from bs4 import BeautifulSoup
import re

# Base URL for the novel chapters on NovelBin
BASE_URL_BIN = "https://novelbin.com/b/pick-me-up-infinite-gacha/chapter-{}"

# Headers to mimic a browser and prevent bot blocking
HEADERS_BIN = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://novelbin.com/b/pick-me-up-infinite-gacha"
    # "Cookie": "YOUR_BROWSER_COOKIES_HERE"  # Add if blocked with 403
}

def clean_text(text):
    """
    Cleans up redundant whitespaces and consecutive empty lines.
    """
    lines = [line.strip() for line in text.split('\n')]
    cleaned_lines = []
    prev_was_empty = False
    for line in lines:
        if not line:
            if not prev_was_empty:
                cleaned_lines.append("")
                prev_was_empty = True
        else:
            cleaned_lines.append(line)
            prev_was_empty = False
    return "\n".join(cleaned_lines)

def fetch_chapter_novelbin(chapter_num, session):
    """
    Fetches a single chapter from NovelBin and returns its title and parsed content.
    """
    url = BASE_URL_BIN.format(chapter_num)
    try:
        response = session.get(url, headers=HEADERS_BIN, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract chapter title
            title = None
            title_elem = soup.select_one('.chr-title, h2.title, h2, .chapter-title')
            if title_elem:
                title = title_elem.get_text(strip=True)
            else:
                title = f"Chapter {chapter_num}"
                
            # Locate the main novel content div (tries multiple selectors)
            content_div = None
            for selector in ['#chr-content', '#chapter-content', '.chr-c', '.chapter-content']:
                content_div = soup.select_one(selector)
                if content_div:
                    break
                    
            if not content_div:
                print(f"Warning: No content div found for chapter {chapter_num}")
                return None, None
                
            raw_text = content_div.get_text()
            cleaned_content = clean_text(raw_text)
            return title, cleaned_content
        else:
            print(f"Failed to load chapter {chapter_num} - Status Code: {response.status_code}")
            return None, None
    except Exception as e:
        print(f"Error fetching chapter {chapter_num}: {e}")
        return None, None

# Configure and run the scraper
start_chapter = 1
end_chapter = 400
output_dir = os.path.join("public", "chapters")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

session = requests.Session()
print(f"Downloading chapters {start_chapter} to {end_chapter} from NovelBin...")

for ch in range(start_chapter, end_chapter + 1):
    title, content = fetch_chapter_novelbin(ch, session)
    
    if title and content:
        # Create a filename-safe title
        safe_title = "".join([c for c in title if c.isalpha() or c.isdigit() or c==' ' or c=='-']).rstrip()
        safe_title = re.sub(r'\s+', ' ', safe_title)
        filename = f"{ch:03d}_{safe_title.replace(' ', '_')}.md"
        filepath = os.path.join(output_dir, filename)
        
        # Format as Markdown
        markdown_content = f"# {title}\n\n{content}\n"
        
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(markdown_content)
        print(f"Saved Chapter {ch}: {filename}")
    else:
        print(f"Skipped Chapter {ch} due to an error.")
        
    # Respectful delay between 1.5 and 3.0 seconds to avoid Cloudflare/anti-bot triggers
    delay = random.uniform(1.5, 3.0)
    time.sleep(delay)

print("Finished downloading all chapters from NovelBin!")